# Análise Exploratória das Métricas Locais de Conectividade

Este notebook apresenta a análise exploratória das métricas locais (por nó)
extraídas das redes funcionais cerebrais obtidas a partir de sinais de EEG
em estado de repouso.

Diferentemente das métricas globais, as métricas locais permitem investigar
a heterogeneidade topológica entre diferentes regiões cerebrais, sendo
potencialmente mais informativas para tarefas de caracterização de grupos
e aprendizado de máquina.

Objetivos desta etapa:
- analisar a distribuição das métricas locais por nó;
- investigar diferenças entre condições experimentais (EO vs EC);
- avaliar a variabilidade intersujeitos;
- preparar as features para a etapa de redução de dimensionalidade (PCA).


In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_dark"


### Carregando o dataset

In [2]:
DATASET_PATH = "../../data/processed/dataset_features.parquet"

df = pd.read_parquet(DATASET_PATH)

df.shape


(108, 409)

In [3]:
df

,degree_0,degree_1,degree_2,degree_3,degree_4,degree_5,degree_6,degree_7,degree_8,degree_9,...,age_60-65,age_65-70,age_70-75,age_75-80,gender_2,handedness_left,handedness_right,education_gymnasium,education_realschule,relationship_status_yes
0,14786.0,14223.0,13305.0,12562.0,9459.0,13334.0,11893.0,8054.0,8195.0,15173.0,...,False,False,False,False,True,False,True,True,False,False
1,20374.0,20232.0,24459.0,20399.0,15591.0,16955.0,16395.0,15200.0,13285.0,22416.0,...,False,False,False,False,True,False,True,True,False,False
2,22982.0,24351.0,22231.0,17394.0,14843.0,20348.0,21595.0,21675.0,12499.0,16804.0,...,False,False,False,False,True,False,True,False,True,True
3,8682.0,9045.0,16592.0,12541.0,8066.0,11202.0,15684.0,15513.0,8463.0,10332.0,...,False,False,False,False,True,False,True,False,True,True
4,29168.0,28481.0,26967.0,27563.0,27400.0,24913.0,23154.0,24220.0,20045.0,14112.0,...,False,False,False,False,False,True,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,12897.0,10346.0,15237.0,15322.0,11459.0,13496.0,11976.0,17013.0,14137.0,13124.0,...,False,False,False,False,True,True,False,True,False,True
104,21065.0,16982.0,22893.0,21942.0,16863.0,23027.0,17824.0,24175.0,19089.0,15923.0,...,False,False,False,False,True,False,True,True,False,False
105,21880.0,19799.0,24711.0,16700.0,10544.0,16339.0,19149.0,25439.0,12665.0,14468.0,...,False,False,False,False,True,False,True,True,False,False
106,21049.0,18595.0,23879.0,19529.0,11742.0,13766.0,19801.0,22879.0,12783.0,10114.0,...,False,False,False,False,True,False,True,True,False,True


### métricas globais × métricas locais

In [4]:
GLOBAL_FEATURES = [
    "global_efficiency",
    "edges_mean",
    "edges_std",
    "edges_cv"
]

NODE_FEATURE_PREFIXES = [
    "degree_",
    "clustering_",
    "betweenness_",
    "local_efficiency_",
    "hub_frequency_",
    "weighted_degree_"
]


In [5]:
NODE_FEATURES = [
    c for c in df.columns
    if any(c.startswith(p) for p in NODE_FEATURE_PREFIXES)
]


In [6]:
len(NODE_FEATURES)


366

DataFrame apenas com métricas locais

In [7]:
df_node = df[["subject_id", "condition"] + NODE_FEATURES].copy()
df_node.shape


(108, 368)

Distribuição de Métricas Locais

In [9]:
degree_cols = [c for c in NODE_FEATURES if c.startswith("degree_")]

df_degree_long = df_node.melt(
    id_vars=["subject_id", "condition"],
    value_vars=degree_cols,
    var_name="node",
    value_name="degree"
)


In [11]:
df_degree_long

,subject_id,condition,node,degree
0,sub-010069,EC,degree_0,14786.0
1,sub-010069,EO,degree_0,20374.0
2,sub-010070,EC,degree_0,22982.0
3,sub-010070,EO,degree_0,8682.0
4,sub-010079,EC,degree_0,29168.0
...,...,...,...,...
6583,sub-010315,EO,degree_60,NaN
6584,sub-010316,EC,degree_60,NaN
6585,sub-010316,EO,degree_60,30184.0
6586,sub-010318,EC,degree_60,NaN


In [12]:
fig = px.box(
    df_degree_long,
    x="node",
    y="degree",
    color="condition",
    title="Distribuição do Grau por Nó e Condição",
    points=False
)
fig.update_layout(xaxis_tickangle=45)
fig.show()


### Variabilidade intersujeitos (importante para ML)

In [13]:
degree_stats = (
    df_degree_long
    .groupby(["node", "condition"])["degree"]
    .agg(["mean", "std"])
    .reset_index()
)


In [14]:
fig = px.bar(
    degree_stats,
    x="node",
    y="std",
    color="condition",
    barmode="group",
    title="Variabilidade Inter-Sujeitos do Grau por Nó"
)
fig.update_layout(xaxis_tickangle=45)
fig.show()


### Heatmap sujeito × nó (padrões globais)

In [15]:
df_degree_matrix = df_node[degree_cols].copy()
df_degree_matrix.index = df_node["subject_id"]


In [16]:
fig = px.imshow(
    df_degree_matrix,
    aspect="auto",
    title="Heatmap do Grau por Nó (Sujeito × Nó)",
    color_continuous_scale="Viridis"
)
fig.show()


df_degree_long.groupby(["node", "condition"])["degree"].describe()


In [17]:
df_degree_long.groupby(["node", "condition"])["degree"].describe()


count          mean          std     min       25%  \
node      condition                                                       
degree_0  EC          54.0  18719.481481  6790.885070  7573.0  14834.75   
          EO          54.0  19016.870370  6105.720992  5829.0  14835.75   
degree_1  EC          54.0  17804.481481  6704.951518  5477.0  14023.75   
          EO          54.0  18257.722222  6433.817082  5968.0  13851.25   
degree_10 EC          54.0  21058.259259  7782.166707  6506.0  15457.75   
...                    ...           ...          ...     ...       ...   
degree_7  EO          54.0  21451.629630  6711.524870  9726.0  16436.25   
degree_8  EC          54.0  15975.388889  5624.505833  7218.0  12593.25   
          EO          54.0  16649.314815  5394.787900  6388.0  13133.50   
degree_9  EC          54.0  15440.648148  5781.034040  5846.0  11401.75   
          EO          54.0  16425.907407  5553.836567  6973.0  12907.25   

                         50%       75%      max  
node      condition                              
degree_0  EC         17556.0  21061.00  43654.0  
          EO         19227.5  22643.50  32007.0  
degree_1  EC         16597.5  20542.50  44186.0  
          EO         18507.5  21303.25  37005.0  
degree_10 EC         20075.5  25552.25  49188.0  
...                      ...       ...      ...  
degree_7  EO         21025.5  25505.00  35840.0  
degree_8  EC         15540.5  19169.25  37357.0  
          EO         16243.5  19785.50  30436.0  
degree_9  EC         15292.0  17668.25  40529.0  
          EO         15688.0  20007.75  29794.0  

[122 rows x 8 columns]